[← 06 - Window Functions, Part 1](<06-window-functions-part-1.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 07 - Window Functions, Part 2: the frame you were already using

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/07-window-functions-part-2.ipynb)

`06 - Window Functions, Part 1` ended with a running total of each customer's orders, where Ada's total read 40 after her first order and 100 after her second, and it explained half of why: adding `ORDER BY` inside `OVER` turns a flat total into a running one. What it didn't say is which rows `SUM()` was adding up for each row. That question has a precise answer with a name, the **frame**, and once you can name it you can change it, and a few results that would otherwise look like bugs stop being mysterious.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** say which rows a window aggregate reads for each row, write `ROWS` and `RANGE` frames on purpose, build moving averages and handle their short first rows, split rows into buckets and percentiles with `NTILE()`, `PERCENT_RANK()`, and `CUME_DIST()`, and fix the `LAST_VALUE()` result that silently copies the current row.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 7 of the All About Data & More curriculum](https://aaboutdata.substack.com/p/sql-window-functions-part-2-the-frame), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks, which only section 10 uses and which has its own one-step setup there.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. The same customers and orders

Four customers (Ada, Bola, Chidi, and Dami, who has never ordered) and five orders between them, the same data as `04` to `06`. Run the setup once; for a clean start later, run `%sql_reset` in a new cell and then run this cell again.

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 2. The frame `06` never named

An aggregate used as a window function, like `SUM()`, `AVG()`, or `COUNT()` with `OVER` after it, works over a **frame** for every row: the specific set of rows inside that row's partition that it reads. `PARTITION BY` decides the group; the frame decides how much of that group gets read for each row.

Not every window function cares about frames. The [SQLite window functions documentation](https://www.sqlite.org/windowfunctions.html) says most built-in window functions ignore the frame entirely, the exceptions being `FIRST_VALUE()`, `LAST_VALUE()`, and `NTH_VALUE()`. `ROW_NUMBER()` and `LAG()` from `06` always look at the whole partition, and so do `NTILE()`, `PERCENT_RANK()`, and `CUME_DIST()` later in this notebook. The frame is something you think about for aggregates and for `FIRST_VALUE()` and `LAST_VALUE()`.

When you put `ORDER BY` inside `OVER` and don't write a frame yourself, the same documentation states the default is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`: everything from the start of the partition up to the current row, plus any rows tied with it. So `06`'s running total, written out in full, is this:

In [3]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_id
        RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM orders
ORDER BY
    customer_id,
    order_id;

customer_id,order_id,amount,running_total
1,101,40,40
1,102,60,100
2,103,80,80
2,104,90,170
3,105,140,140


Exactly the same five rows as `06`: 40 and 100 for Ada, 80 and 170 for Bola, and 140 for Chidi. The frame clause was always there; you just weren't the one writing it.

That also explains the other half of `06`. `SUM(amount) OVER (PARTITION BY customer_id)`, with no `ORDER BY`, gave both of Ada's rows a flat 100. Without an `ORDER BY`, every row in the partition counts as a **peer** of every other, and peers always share a frame, so each row's frame is the whole partition. Adding `ORDER BY` is what gave the rows an order to stop at.

<p align="center">
  <img src="graphics/07_frame_options.png" width="720" alt="Four frames: no ORDER BY reads the whole partition; ORDER BY with no frame reads RANGE from the start to the current row and its peers; ROWS 1 PRECEDING reads this row and the one before; UNBOUNDED both ways reads the whole partition in order. ROW_NUMBER, LAG, LEAD, NTILE, PERCENT_RANK and CUME_DIST ignore the frame">
</p>

## 3. Common mistake: a running total over a column with ties

The word `RANGE` in that default matters, and `06`'s exercise never exposed it, because `order_id` is unique and no two rows ever tied. Here's a case where they do. Suppose you want a company-wide running total of revenue, building up customer by customer, so you order by `customer_id`:

In [4]:
%%sql
SELECT
    order_id,
    customer_id,
    amount,
    SUM(amount) OVER (
        ORDER BY customer_id
    ) AS default_range,
    SUM(amount) OVER (
        ORDER BY customer_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS rows_frame
FROM orders
ORDER BY
    customer_id,
    order_id;

order_id,customer_id,amount,default_range,rows_frame
101,1,40,100,40
102,1,60,100,100
103,2,80,270,180
104,2,90,270,270
105,3,140,410,410


The `default_range` column doesn't run at all for Ada; it jumps straight to 100 on her first order. Ada's two orders both have `customer_id = 1`, so under `RANGE` they're peers, and for a `RANGE` frame, `CURRENT ROW` includes the current row's peers too. Order 101's frame therefore already contains order 102. `ROWS` counts individual rows instead, so its frame for order 101 stops at order 101.

<p align="center">
  <img src="graphics/07_rows_vs_range_ties.png" width="520" alt="Ordered by customer_id, the default RANGE frame gives orders 101 and 102 the same total of 100 because they tie; a ROWS frame ordered by customer_id and order_id builds 40, 100, 180, 270, 410">
</p>

This is a silent mistake. Nothing errors, and the final total of 410 is right in both columns, so a glance at the bottom row tells you nothing. It shows up in real data the moment you run a running total by date and two orders share a date, which section 10 shows happening 43 times on a single day.

### Switching to ROWS isn't the whole fix

It's tempting to conclude that `ROWS` is simply correct. Look at what it did, though: it counted order 101 before order 102, but nothing in `ORDER BY customer_id` asked for that. The [SQLite SELECT documentation](https://www.sqlite.org/lang_select.html) says the order of two rows whose `ORDER BY` values are all equal is undefined, so under `ROWS`, 40 on order 101 is one valid answer, and 60 there would have been another.

The real fix is to stop the tie from existing, by adding a column that makes the ordering unique:

In [5]:
%%sql
SELECT
    order_id,
    customer_id,
    amount,
    SUM(amount) OVER (
        ORDER BY
            customer_id,
            order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM orders
ORDER BY
    customer_id,
    order_id;

order_id,customer_id,amount,running_total
101,1,40,40
102,1,60,100
103,2,80,180
104,2,90,270
105,3,140,410


40, 100, 180, 270, 410, on purpose this time. With a unique `ORDER BY`, no row has any peers, so `ROWS` and the default `RANGE` agree. Writing `ROWS` anyway is still worth it, since it tells the next person reading the query that the frame was a decision rather than an accident.

## 4. Moving windows: a frame that slides

The frame doesn't have to start at the beginning of the partition. `ROWS BETWEEN 1 PRECEDING AND CURRENT ROW` means "this row and the one before it", which gives you a **moving average**, here across all five orders in order:

In [6]:
%%sql
SELECT
    order_id,
    amount,
    AVG(amount) OVER (
        ORDER BY order_id
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS two_order_avg,
    COUNT(*) OVER (
        ORDER BY order_id
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS rows_in_frame
FROM orders
ORDER BY
    order_id;

order_id,amount,two_order_avg,rows_in_frame
101,40,40.0,1
102,60,50.0,2
103,80,70.0,2
104,90,85.0,2
105,140,115.0,2


Order 104's average is 85, the mean of 80 and 90, and order 105's is 115, the mean of 90 and 140. The window moves forward one row at a time and drops the oldest row as it goes.

### Edge case: what the first row averages

Order 101 has no row before it, so its frame holds one row, and its "two-order average" is 40, the average of 40 by itself. It isn't `NULL`, the way `LAG()` returned `NULL` in `06`, and it isn't 20, which is what you'd get by treating a missing row as zero. The frame gets smaller at the edge, and `AVG()` averages whatever is inside it.

That's why the `rows_in_frame` column is there. With a seven-row frame (`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW`), the first six rows are averages of fewer than seven values, and a chart of them can show an early spike or dip that's only an artifact of a short frame. Counting the rows in the frame lets you label those rows, or drop them with `WHERE rows_in_frame = 7` from an outer query, the same compute-then-filter move `06` needed for `rn = 1`.

The other caveat with moving windows is that `ROWS` counts rows, not time. On daily data, `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` covers seven rows, which is seven days only if every day has exactly one row. A missing day, or a day with two rows, quietly changes what the window covers.

## 5. NTILE(): splitting rows into buckets

`NTILE(n)` splits a partition into `n` buckets, as evenly as the row count allows, in the order you give it. It's what sits behind labels like "top half" or "bottom quartile". Here's every customer split into two spending halves, with Dami's zero included through a `LEFT JOIN`:

In [7]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
)
SELECT
    customer_name,
    total_spent,
    NTILE(2) OVER (ORDER BY total_spent DESC) AS spend_half
FROM customer_totals
ORDER BY
    total_spent DESC;

customer_name,total_spent,spend_half
Bola,170,1
Chidi,140,1
Ada,100,2
Dami,0,2


Four customers split cleanly into two buckets of two. `ORDER BY total_spent DESC` is what makes bucket 1 the top spenders; flip it to ascending and bucket 1 becomes the bottom half. Hold onto the word "cleanly", because exercise 1 asks what happens when the count doesn't divide evenly.

## 6. PERCENT_RANK() vs CUME_DIST()

`NTILE()` makes you pick a number of buckets up front. `PERCENT_RANK()` and `CUME_DIST()` don't; each gives a row's relative position as a number between 0 and 1. They look interchangeable, and they aren't (the results are rounded to two places here):

In [8]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
)
SELECT
    customer_name,
    total_spent,
    ROUND(PERCENT_RANK() OVER (ORDER BY total_spent), 2) AS pct_rank,
    ROUND(CUME_DIST() OVER (ORDER BY total_spent), 2) AS cume_dist
FROM customer_totals
ORDER BY
    total_spent;

customer_name,total_spent,pct_rank,cume_dist
Dami,0,0.0,0.25
Ada,100,0.33,0.5
Chidi,140,0.67,0.75
Bola,170,1.0,1.0


`PERCENT_RANK()` answers "what share of the *other* rows rank below this one?" The formula is (rank - 1) divided by (rows - 1), where tied rows share a rank, so the lowest row is always 0 and the highest is always 1. Ada ranks second of four, so her value is 1/3.

`CUME_DIST()` answers "what share of *all* rows sit at or below this one?" That's the row's position, counting any rows tied with it, divided by the total row count, so the lowest row can never be 0, since it always counts itself. Dami is the only row at or below 0, one row out of four, so she gets 0.25.

Ada shows the difference most clearly: she's at 0.33 by one measure and 0.5 by the other, and "Ada is at the 33rd percentile" and "Ada is at the 50th percentile" are both defensible sentences about the same customer. Before either number goes into a report, decide which question you're answering.

## 7. Common mistake: trusting LAST_VALUE() with the default frame

`FIRST_VALUE()` and `LAST_VALUE()` sound like a matched pair: the first and last value in each customer's orders. Try both with the default frame:

In [9]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    FIRST_VALUE(amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_id
    ) AS first_order,
    LAST_VALUE(amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_id
    ) AS last_order
FROM orders
ORDER BY
    customer_id,
    order_id;

customer_id,order_id,amount,first_order,last_order
1,101,40,40,40
1,102,60,40,60
2,103,80,80,80
2,104,90,80,90
3,105,140,140,140


`first_order` is right on every row. `last_order` is just `amount` copied over, so on order 101 it says Ada's last order was 40, when it was 60.

Nothing is wrong with `LAST_VALUE()` itself. It returns the value from the last row in the *frame*, and the default frame runs from the start of the partition to the current row, so the last row of that frame is the current row, every time. `FIRST_VALUE()` only looks correct because every default frame starts at the beginning of the partition.

<p align="center">
  <img src="graphics/07_last_value_frame_trap.png" width="600" alt="For order 101, the default frame holds only order 101, so LAST_VALUE returns 40; with ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING the frame holds orders 101 and 102, so it returns 60">
</p>

The fix is to widen the frame to the whole partition in both directions:

In [10]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    LAST_VALUE(amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS last_order
FROM orders
ORDER BY
    customer_id,
    order_id;

customer_id,order_id,amount,last_order
1,101,40,60
1,102,60,60
2,103,80,90
2,104,90,90
3,105,140,140


Now both of Ada's rows show 60 and both of Bola's show 90. Chidi was never a useful test of this, because a one-order partition gives the same answer either way, and that's part of why the mistake survives: any partition where the current row happens to be the last row hides it.

## 8. Every window has edges

Every running total, moving average, and `LAST_VALUE()` you write reads a frame, whether you write the frame or not. `PARTITION BY` decides the group, `ORDER BY` decides the sequence, and the frame decides where the slice read for each row starts and stops. The two silent mistakes here, the running total that jumps on ties and the `LAST_VALUE()` that hands back the current row, come from the same place: a default frame that wasn't the one you assumed. When an aggregate or `LAST_VALUE()` result looks slightly off, check the frame before anything else.

## 9. Try it yourself

Write each query in a new cell before looking at the [solutions notebook](<solutions/07-window-functions-part-2-solutions.ipynb>), predicting the result first.

1. **Four customers, three buckets.** Change the `NTILE(2)` query from section 5 to `NTILE(3)`. Four customers don't split evenly into three buckets, so one bucket has to hold two. Before you run it, predict which bucket gets the extra customer, and which bucket Dami ends up in.
2. **A centered moving average.** Across all five orders in `order_id` order, average each order with the one before it *and* the one after it. Add a column counting the rows in each frame. Which rows average fewer than three orders?
3. **Distance from the last order.** For each order, show how much smaller it is than that customer's most recent order (a negative number if it's smaller, zero for the most recent order itself).
4. **Stretch, on AdventureWorks.** Using monthly sales (section 10 shows how to build them), compute a three-month moving average, and show it only for the months of 2012, with every 2012 month averaging three full months.

## 10. Scale it up: frames in AdventureWorks

Five orders make every frame checkable by hand, and real data is where ties and short frames hide inside numbers nobody checks row by row. This section runs the lesson's ideas against **AdventureWorks2022**, Microsoft's sample database for an invented bike company, on SQL Server. Frames work the same way there, with one restriction you'll meet below.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [11]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### A running total by date, where every row ties

Here's the most common running total there is, total sales over time, ordered by `OrderDate`. The first day of data had 43 orders:

In [12]:
%%tsql
SELECT TOP 6
    SalesOrderID,
    OrderDate,
    TotalDue,
    SUM(TotalDue) OVER (ORDER BY OrderDate) AS default_range,
    SUM(TotalDue) OVER (
        ORDER BY OrderDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS rows_frame
FROM Sales.SalesOrderHeader
ORDER BY
    OrderDate,
    SalesOrderID;

SalesOrderID,OrderDate,TotalDue,default_range,rows_frame
43659,2011-05-31 00:00:00,23153.2339,567020.9498,23153.2339
43660,2011-05-31 00:00:00,1457.3288,567020.9498,24610.5627
43661,2011-05-31 00:00:00,36865.8012,567020.9498,61476.3639
43662,2011-05-31 00:00:00,32474.9324,567020.9498,93951.2963
43663,2011-05-31 00:00:00,472.3108,567020.9498,94423.6071
43664,2011-05-31 00:00:00,27510.4109,567020.9498,121934.0180


Every one of the 43 orders on 31 May 2011 shows the same `default_range` of about $567,021, the whole day's sales, because they all tie on `OrderDate`, which is section 3's mistake on real data. Which one is right depends on the question: if you want a running total *per day*, the `RANGE` number is exactly the day's closing total, and if you want it per order, you need `ROWS` and a tie-breaker:

In [13]:
%%tsql
SELECT TOP 6
    SalesOrderID,
    OrderDate,
    TotalDue,
    SUM(TotalDue) OVER (
        ORDER BY
            OrderDate,
            SalesOrderID
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM Sales.SalesOrderHeader
ORDER BY
    OrderDate,
    SalesOrderID;

SalesOrderID,OrderDate,TotalDue,running_total
43659,2011-05-31 00:00:00,23153.2339,23153.2339
43660,2011-05-31 00:00:00,1457.3288,24610.5627
43661,2011-05-31 00:00:00,36865.8012,61476.3639
43662,2011-05-31 00:00:00,32474.9324,93951.2963
43663,2011-05-31 00:00:00,472.3108,94423.6071
43664,2011-05-31 00:00:00,27510.4109,121934.0180


### A restriction SQLite doesn't have

SQLite lets a `RANGE` frame use a numeric offset, like `RANGE BETWEEN 1 PRECEDING AND CURRENT ROW`, meaning "rows whose `ORDER BY` value is within 1 of this one". SQL Server doesn't. This cell is meant to fail:

In [14]:
%%tsql --expect-error
SELECT TOP 3
    SalesOrderID,
    SUM(TotalDue) OVER (
        ORDER BY SalesOrderID
        RANGE BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS nearby_total
FROM Sales.SalesOrderHeader;

Error (this cell is meant to fail): RANGE is only supported with UNBOUNDED and CURRENT ROW window frame delimiters.


On SQL Server, `RANGE` only works with `UNBOUNDED PRECEDING`, `CURRENT ROW`, and `UNBOUNDED FOLLOWING`. For a sliding window there, use `ROWS`.

### A moving average of monthly sales

Monthly sales jump around, so a three-month moving average smooths them. `rows_in_frame` shows the short frames at the start:

In [15]:
%%tsql
WITH monthly_sales AS (
    SELECT
        DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1) AS month_start,
        SUM(TotalDue) AS sales
    FROM Sales.SalesOrderHeader
    GROUP BY
        DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1)
)
SELECT TOP 5
    month_start,
    sales,
    AVG(sales) OVER (
        ORDER BY month_start
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS three_month_avg,
    COUNT(*) OVER (
        ORDER BY month_start
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rows_in_frame
FROM monthly_sales
ORDER BY
    month_start;

month_start,sales,three_month_avg,rows_in_frame
2011-05-01,567020.9498,567020.9498,1
2011-06-01,507096.4690,537058.7094,2
2011-07-01,2292182.8828,1122100.1005,3
2011-08-01,2800576.1723,1866618.5080,3
2011-09-01,554791.6082,1882516.8877,3


May 2011's "three-month average" is just May, and June's is May and June. A chart of this would start with two numbers that aren't what the label says, which is section 4's edge case on real data.

### Customer quartiles with NTILE()

Split the 19,119 customers who've ordered into four quartiles by total spending, then summarize each quartile:

In [16]:
%%tsql
WITH customer_totals AS (
    SELECT
        CustomerID,
        SUM(TotalDue) AS total_spent
    FROM Sales.SalesOrderHeader
    GROUP BY
        CustomerID
),
quartiles AS (
    SELECT
        total_spent,
        NTILE(4) OVER (ORDER BY total_spent DESC) AS quartile
    FROM customer_totals
)
SELECT
    quartile,
    COUNT(*) AS customers,
    MIN(total_spent) AS smallest,
    MAX(total_spent) AS largest,
    SUM(total_spent) AS quartile_total
FROM quartiles
GROUP BY
    quartile
ORDER BY
    quartile;

quartile,customers,smallest,largest,quartile_total
1,4780,3119.1494,989184.0820,114013491.0078
2,4780,606.6229,3119.1494,8406689.9822
3,4780,60.7529,606.6229,653697.8598
4,4779,1.5183,60.7419,142907.2661


19,119 doesn't divide by 4, so the first three quartiles get 4,780 customers and the last gets 4,779, "larger groups first" again. The top quartile spent about $114 million of the roughly $123 million total, which is the kind of finding quartiles exist to surface.

Look closely at the boundary between quartiles 1 and 2: $3,119.15 is the smallest total in quartile 1 *and* the largest in quartile 2. Three customers spent exactly that amount, and `NTILE()` split them across two buckets, because it fills buckets by row count, not by value. If identical customers must land in the same group, `NTILE()` is the wrong tool, and a `CUME_DIST()` threshold is the usual alternative, since tied rows always get the same `CUME_DIST()`.

## Before you trust a window frame

- If there's an `ORDER BY` inside `OVER` and no frame written, the frame is `RANGE` from the start of the partition to the current row and its peers.
- Can two rows tie on the `ORDER BY`? If so, decide whether they should be counted together (`RANGE`) or one by one (`ROWS` plus a tie-breaker).
- Count the rows in a moving window, and label or drop the short frames at the edge.
- `ROWS` counts rows, not days. Check for missing or doubled dates before calling a seven-row window a week.
- Give `LAST_VALUE()` an explicit frame that reaches the end of the partition.
- `NTILE()` splits ties across buckets; `CUME_DIST()` keeps them together.

## Go deeper

- **[Window Functions](https://www.sqlite.org/windowfunctions.html)**, SQLite. The precise frame rules, including peers, `EXCLUDE`, and the "larger groups occur first" rule for `ntile()`.
- **[Window function calls](https://www.postgresql.org/docs/current/sql-expressions.html)** and **[Window functions](https://www.postgresql.org/docs/current/functions-window.html)**, PostgreSQL. A careful explanation of `ROWS`, `RANGE`, and `GROUPS` frames, and of why `LAST_VALUE()` behaves the way it does.
- **[OVER clause](https://learn.microsoft.com/en-us/sql/t-sql/queries/select-over-clause-transact-sql)**, **[NTILE](https://learn.microsoft.com/en-us/sql/t-sql/functions/ntile-transact-sql)**, **[PERCENT_RANK](https://learn.microsoft.com/en-us/sql/t-sql/functions/percent-rank-transact-sql)**, **[CUME_DIST](https://learn.microsoft.com/en-us/sql/t-sql/functions/cume-dist-transact-sql)**, and **[LAST_VALUE](https://learn.microsoft.com/en-us/sql/t-sql/functions/last-value-transact-sql)**, Microsoft Learn. The SQL Server references behind section 10, including the `RANGE` restriction.
- **Practice:** [SQLZoo: Window functions](https://sqlzoo.net/wiki/Window_functions) and [DataLemur's window functions tutorial](https://datalemur.com/sql-tutorial/sql-aggregate-window-functions); running totals and moving averages are among the most common SQL interview questions.

For interview questions on this topic with model answers, see the [study guide](<study-guides/07-window-functions-part-2.md>).

## What's next

Every query since `04` has read from the `orders` table without ever asking where its rows came from. `08a - Changing Data, Part 1` answers that, with the statements that create a table, put rows into it, and change or delete them: `CREATE TABLE`, `INSERT`, `UPDATE`, and `DELETE`.

---

[← 06 - Window Functions, Part 1](<06-window-functions-part-1.ipynb>) · [Course Overview](<00-course-overview.ipynb>) · [08a - Changing Data, Part 1 →](<08a-changing-data-part-1.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).